# Notebook 07 — DeepAR (C3) + TFT (C0) on GPU

**Run this on Google Colab with A100 GPU (~30 min total).**

What this notebook does:
1. Installs `neuralforecast` and uploads the three data parquets
2. Runs **DeepAR** on C3 (Sparse Long-Tail, 1017 SKUs)
3. Runs **TFT** on C0 (Erratic, 515 SKUs)
4. Evaluates both with the same metrics as notebooks 05/06
5. Saves prediction parquets for download → copy into `data/primary_models/`

**Baselines to beat (from notebook 05):**
- C3 best baseline: iMAPA WMAPE = 110.5%
- C0 best baseline: TSB WMAPE = 108.0%; our LGBM = 85.9%

In [ ]:
# ── Install (Colab only) ──────────────────────────────────────────────────────
!pip install neuralforecast -q

In [ ]:
# Upload these files from your local data/ folder:
#   daily_train_clustered_winsorized.parquet  (used for model training)
#   daily_val_clustered.parquet               (used for early stopping validation)
#   daily_test_clustered.parquet              (used for evaluation against true values)

from google.colab import files
uploaded = files.upload()  # select all three parquets


In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import time
from pathlib import Path

import torch
print(f'GPU available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'Device: {torch.cuda.get_device_name(0)}')

from neuralforecast import NeuralForecast
from neuralforecast.models import DeepAR, TFT
from neuralforecast.losses.pytorch import DistributionLoss, MQLoss
# MQLoss: TFT training loss (predicts 10/50/90th percentile)
# DistributionLoss(NegativeBinomial): DeepAR training loss
# WMAPE is computed as an evaluation metric below using our own wmape() function

OUT = Path('predictions')
OUT.mkdir(exist_ok=True)


In [ ]:
# ── Load data ─────────────────────────────────────────────────────────────────
# Winsorized train: caps per-product p99 spikes → cleaner training loss
# Unwinsorized val/test: evaluate against true values for apples-to-apples comparison
train_raw = pd.read_parquet('daily_train_clustered_winsorized.parquet')
val_raw   = pd.read_parquet('daily_val_clustered.parquet')
test_raw  = pd.read_parquet('daily_test_clustered.parquet')

# Combine winsorized train + unwinsorized val as history for fitting
# Val is used by neuralforecast for early stopping (val_size=H), not for training loss
history = pd.concat([train_raw, val_raw], ignore_index=True)
test    = test_raw.copy()

H = test.date.nunique()  # forecast horizon = 157 days
print(f'History: {history.date.min().date()} → {history.date.max().date()}')
print(f'Test:    {test.date.min().date()} → {test.date.max().date()}  (H={H} days)')

EXOG_COLS = ['days_to_holiday', 'is_christmas_week',
             'dow_sin', 'dow_cos', 'week_sin', 'week_cos', 'month_sin', 'month_cos']

def make_nf_df(df):
    return (
        df.rename(columns={'product_family_name': 'unique_id',
                           'date': 'ds',
                           'total_sales': 'y'})
          .sort_values(['unique_id', 'ds'])
          .reset_index(drop=True)
    )


In [ ]:
# ── Metric functions (same as notebooks 05/06) ────────────────────────────────
def wmape(y, p, eps=1.0):
    return np.sum(np.abs(y-p)) / max(np.sum(np.maximum(np.abs(y), eps)), eps) * 100

def eps_mape(y, p, eps=1.0):
    return np.mean(np.abs(y-p) / np.maximum(np.abs(y), eps)) * 100

def nz_mape(y, p):
    mask = y > 0
    return np.mean(np.abs(y[mask]-p[mask]) / y[mask]) * 100 if mask.sum() > 0 else np.nan

def score(df, pred_col, actual_col='y'):
    y = df[actual_col].fillna(0).values
    p = np.maximum(df[pred_col].fillna(0).values, 0)
    return {
        'WMAPE':    round(wmape(y, p), 2),
        'eps_MAPE': round(eps_mape(y, p), 2),
        'NZ_MAPE':  round(nz_mape(y, p), 2),
        'zero_rate%': round((y == 0).mean() * 100, 1),
    }

## 1-Epoch Timing Test
Run this first to confirm GPU is active and estimate full training time.

In [ ]:
# Quick 1-epoch timing test on C3 DeepAR
c3_hist = make_nf_df(history[history.cluster == 3])

test_model = DeepAR(
    h=H,
    input_size=28,
    lstm_n_layers=2,
    lstm_hidden_size=64,
    trajectory_samples=100,
    loss=DistributionLoss(distribution='NegativeBinomial', level=[80, 95]),
    max_steps=1,
    accelerator='gpu',
)
t0 = time.time()
NeuralForecast(models=[test_model], freq='D').fit(c3_hist)
per_step = time.time() - t0
print(f'1 step: {per_step:.1f}s → 200 steps ≈ {per_step*200/60:.1f} min (DeepAR C3)')

## DeepAR — C3 (Sparse Long-Tail, 1017 SKUs)

- NegativeBinomial output — handles zero-inflated count data
- Global model: all 1017 C3 SKUs trained jointly
- Exogenous regressors: holiday + cyclical calendar features
- Outputs 80% and 95% prediction intervals (free from distributional output)

**Baseline to beat**: iMAPA WMAPE = 110.5%

In [ ]:
c3_hist = make_nf_df(history[history.cluster == 3])
c3_test = make_nf_df(test[test.cluster == 3])

futr_cols = EXOG_COLS

deepar = DeepAR(
    h=H,
    input_size=56,
    lstm_n_layers=2,
    lstm_hidden_size=128,
    trajectory_samples=200,
    futr_exog_list=futr_cols,
    loss=DistributionLoss(distribution='NegativeBinomial', level=[80, 95]),
    learning_rate=1e-3,
    max_steps=200,
    val_check_steps=25,
    early_stop_patience_steps=4,
    accelerator='gpu',
    batch_size=32,
    scaler_type='standard',
)

nf_c3 = NeuralForecast(models=[deepar], freq='D')

t0 = time.time()
nf_c3.fit(c3_hist, val_size=H)
print(f'Training time: {(time.time()-t0)/60:.1f} min')


In [ ]:
t0 = time.time()

# Get exact future dates neuralforecast expects
future_df = nf_c3.make_future_dataframe(df=c3_hist)

# Build exog lookup from ALL dates (train+val+test) to avoid nulls
all_dates = pd.concat([
    train_raw[train_raw.cluster==3][['product_family_name','date']+EXOG_COLS],
    val_raw[val_raw.cluster==3][['product_family_name','date']+EXOG_COLS],
    test_raw[test_raw.cluster==3][['product_family_name','date']+EXOG_COLS],
])
all_dates = all_dates.rename(columns={'product_family_name':'unique_id','date':'ds'})
future_df = future_df.merge(all_dates, on=['unique_id','ds'], how='left')

# Fill any remaining nulls using forward-fill within each SKU
# (handles edge dates not in the dataset grid)
future_df = future_df.sort_values(['unique_id','ds'])
future_df[EXOG_COLS] = future_df.groupby('unique_id')[EXOG_COLS].ffill().bfill()

null_check = future_df[EXOG_COLS].isnull().sum().sum()
print(f'Null values after fill: {null_check}')

c3_preds = nf_c3.predict(futr_df=future_df)
print(f'Prediction time: {time.time()-t0:.1f}s')
print(c3_preds.head())
print(c3_preds.columns.tolist())


In [ ]:
# Merge predictions with actuals and evaluate
c3_eval = c3_test[['unique_id', 'ds', 'y']].merge(
    c3_preds.reset_index(), on=['unique_id', 'ds'], how='left'
)

# Point forecast column (median)
point_col = [c for c in c3_eval.columns if 'DeepAR' in c and 'lo' not in c and 'hi' not in c][0]
print(f'Point forecast column: {point_col}')

metrics = score(c3_eval, point_col)
print('\nDeepAR C3 metrics:')
for k, v in metrics.items():
    print(f'  {k}: {v}')

print('\nBaseline comparison:')
print('  iMAPA WMAPE=110.5%  eps_MAPE=141.4%  NZ_MAPE=117.8%')
print('  TS-HGB (prior team) WMAPE=150.1%')

# Rename columns for clarity and save
c3_eval = c3_eval.rename(columns={
    'unique_id': 'product_family_name',
    'ds': 'date',
    point_col: 'pred_deepar'
})
c3_eval['cluster'] = 3
c3_eval.to_parquet(OUT / 'c3_deepar_predictions.parquet', index=False)
print('\nSaved: predictions/c3_deepar_predictions.parquet')

## TFT — C0 (Erratic, 515 SKUs)

- Temporal Fusion Transformer: multi-head attention over temporal patterns
- Global model: all 515 C0 SKUs trained jointly with `unique_id` as static covariate
- Exogenous regressors: holiday + cyclical calendar features
- Outputs quantile forecasts (10th, 50th, 90th percentile)

**Baselines to beat**: TSB WMAPE=108.0%, our LGBM WMAPE=85.9%

In [ ]:
c0_hist = make_nf_df(history[history.cluster == 0])
c0_test = make_nf_df(test[test.cluster == 0])

tft = TFT(
    h=H,
    input_size=56,
    hidden_size=64,
    n_head=4,
    attn_dropout=0.1,
    dropout=0.1,
    futr_exog_list=futr_cols,
    loss=MQLoss(),
    learning_rate=1e-3,
    max_steps=300,
    val_check_steps=30,
    early_stop_patience_steps=4,
    accelerator='gpu',
    batch_size=32,
    scaler_type='standard',
)

nf_c0 = NeuralForecast(models=[tft], freq='D')

t0 = time.time()
nf_c0.fit(c0_hist, val_size=H)
print(f'Training time: {(time.time()-t0)/60:.1f} min')


In [ ]:
t0 = time.time()

future_df = nf_c0.make_future_dataframe(df=c0_hist)

all_dates = pd.concat([
    train_raw[train_raw.cluster==0][['product_family_name','date']+EXOG_COLS],
    val_raw[val_raw.cluster==0][['product_family_name','date']+EXOG_COLS],
    test_raw[test_raw.cluster==0][['product_family_name','date']+EXOG_COLS],
])
all_dates = all_dates.rename(columns={'product_family_name':'unique_id','date':'ds'})
future_df = future_df.merge(all_dates, on=['unique_id','ds'], how='left')
future_df = future_df.sort_values(['unique_id','ds'])
future_df[EXOG_COLS] = future_df.groupby('unique_id')[EXOG_COLS].ffill().bfill()

null_check = future_df[EXOG_COLS].isnull().sum().sum()
print(f'Null values after fill: {null_check}')

c0_preds = nf_c0.predict(futr_df=future_df)
print(f'Prediction time: {time.time()-t0:.1f}s')
print(c0_preds.head())
print(c0_preds.columns.tolist())


In [ ]:
# Merge predictions with actuals and evaluate
c0_eval = c0_test[['unique_id', 'ds', 'y']].merge(
    c0_preds.reset_index(), on=['unique_id', 'ds'], how='left'
)

point_col = [c for c in c0_eval.columns if 'TFT' in c and 'lo' not in c and 'hi' not in c][0]
print(f'Point forecast column: {point_col}')

metrics = score(c0_eval, point_col)
print('\nTFT C0 metrics:')
for k, v in metrics.items():
    print(f'  {k}: {v}')

print('\nBaseline comparison:')
print('  TSB WMAPE=108.0%  (classical baseline)')
print('  LGBM WMAPE=85.9%  (our Phase 4 model — the real bar to beat)')

c0_eval = c0_eval.rename(columns={
    'unique_id': 'product_family_name',
    'ds': 'date',
    point_col: 'pred_tft'
})
c0_eval['cluster'] = 0
c0_eval.to_parquet(OUT / 'c0_tft_predictions.parquet', index=False)
print('\nSaved: predictions/c0_tft_predictions.parquet')

## Summary

In [ ]:
print('=' * 60)
print('FINAL RESULTS SUMMARY')
print('=' * 60)

results = [
    ('C3', 'iMAPA (baseline)',    110.5, 141.4, 117.8),
    ('C3', 'TS-HGB (prior team)', 150.1, 112.2, 154.4),
    ('C0', 'TSB (baseline)',      108.0, 597.9, 167.6),
    ('C0', 'LGBM (our Phase 4)',   85.9, 217.5,  94.9),
]

# DeepAR C3
c3_r = pd.read_parquet(OUT / 'c3_deepar_predictions.parquet')
m = score(c3_r, 'pred_deepar')
results.insert(1, ('C3', 'DeepAR (ours)', m['WMAPE'], m['eps_MAPE'], m['NZ_MAPE']))

# TFT C0
c0_r = pd.read_parquet(OUT / 'c0_tft_predictions.parquet')
m = score(c0_r, 'pred_tft')
results.insert(-1, ('C0', 'TFT (ours)', m['WMAPE'], m['eps_MAPE'], m['NZ_MAPE']))

summary = pd.DataFrame(results, columns=['Cluster','Model','WMAPE','eps_MAPE','NZ_MAPE'])
print(summary.to_string(index=False))
print()
print('Download predictions/ folder and copy parquets to data/primary_models/')

In [ ]:
# ── Download results ──────────────────────────────────────────────────────────
import shutil
shutil.make_archive('predictions', 'zip', 'predictions')
files.download('predictions.zip')